# MCP 프롬프트 정의와 사용

**Skilljar Lessons 09-10 대응**

이 노트북에서 다루는 내용:
1. `@mcp.prompt()` 데코레이터로 프롬프트 정의
2. 파라미터화된 프롬프트 템플릿
3. 클라이언트에서 프롬프트 목록 조회 및 사용
4. Claude API와 프롬프트 통합

In [ ]:
# ── Setup ──────────────────────────────────────────────
import json
import anthropic
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

mcp = FastMCP("Prompt Demo Server")
MODEL = "claude-haiku-4-5"

## §1. MCP 프롬프트란?

MCP의 3대 기능 중 **사용자가 직접 선택**하는 기능입니다.

| 기능 | 제어 주체 | 비유 |
|------|----------|------|
| Tools | 모델 (LLM) | AI가 알아서 도구 선택 |
| Resources | 애플리케이션 | 개발자가 데이터 제공 |
| **Prompts** | **사용자** | **사용자가 템플릿 선택** |

## §2. 프롬프트 정의

`@mcp.prompt()` 데코레이터로 재사용 가능한 프롬프트 템플릿을 만듭니다.

In [ ]:
# 프롬프트 1: 코드 리뷰
@mcp.prompt()
def review_code(code: str, language: str = "python") -> str:
    """코드 리뷰를 수행하는 프롬프트 템플릿입니다.

    Args:
        code: 리뷰할 코드
        language: 프로그래밍 언어
    """
    return f"""당신은 시니어 {language} 개발자입니다.
다음 코드를 리뷰하고 개선점을 제시해주세요:

```{language}
{code}
```

다음 관점에서 검토해주세요:
1. 코드 품질과 가독성
2. 잠재적 버그
3. 성능 최적화
4. Best practices 준수 여부"""


# 프롬프트 2: 구조 검토
@mcp.prompt()
def structural_review(
    member_type: str,
    width: float,
    depth: float,
    fck: float,
    fy: float
) -> str:
    """구조 부재 설계 검토를 수행하는 프롬프트 템플릿입니다.

    Args:
        member_type: 부재 종류 (beam, column, slab)
        width: 부재 폭 (mm)
        depth: 부재 깊이 (mm)
        fck: 콘크리트 설계기준 압축강도 (MPa)
        fy: 철근 항복강도 (MPa)
    """
    return f"""당신은 건축구조 전문가입니다. KDS 41 17 00 기준에 따라 다음 RC {member_type}의 설계를 검토해주세요.

## 부재 제원
- 종류: {member_type}
- 폭 (b): {width} mm
- 깊이 (d): {depth} mm
- fck: {fck} MPa
- fy: {fy} MPa

## 검토 항목
1. 최소/최대 철근비 확인
2. 휨 강도 검토
3. 전단 강도 검토 (해당 시)
4. 처짐 검토 (해당 시)"""


# 프롬프트 3: 설계 기준 체크리스트
@mcp.prompt()
def design_check(standard: str = "KDS 41 17 00") -> str:
    """설계 기준 체크리스트를 생성하는 프롬프트입니다.

    Args:
        standard: 적용 기준 (기본: KDS 41 17 00)
    """
    return f"""당신은 구조설계 검토자입니다. {standard} 기준에 따른 설계 체크리스트를 작성해주세요.

체크리스트에 포함할 항목:
1. 재료 기준 확인
2. 하중 조합 검토
3. 단면 설계 적정성
4. 상세 설계 확인
5. 시공성 검토"""


print("3개 프롬프트가 등록되었습니다.")

## §3. 프롬프트 목록 조회 및 사용

In [ ]:
import asyncio

async def demo_prompts():
    # 프롬프트 목록 조회
    prompts = await mcp.list_prompts()
    print("=== 등록된 프롬프트 ===")
    for p in prompts:
        print(f"  - {p.name}: {p.description}")
        if p.arguments:
            for arg in p.arguments:
                req = "(필수)" if arg.required else "(선택)"
                print(f"      {arg.name} {req}: {arg.description}")
        print()

    # 프롬프트 호출
    print("=== structural_review 프롬프트 실행 ===")
    result = await mcp.get_prompt(
        "structural_review",
        arguments={
            "member_type": "beam",
            "width": "300",
            "depth": "600",
            "fck": "27",
            "fy": "400"
        }
    )
    for msg in result.messages:
        print(f"[{msg.role}]")
        print(msg.content.text)
        print()

await demo_prompts()

## §4. Claude API와 프롬프트 통합

MCP 프롬프트에서 생성된 메시지를 Claude API에 직접 전달합니다.

In [ ]:
async def use_prompt_with_claude(prompt_name: str, arguments: dict):
    """MCP 프롬프트를 Claude API 호출에 활용합니다."""
    api_client = anthropic.Anthropic()

    # 1. MCP 프롬프트에서 메시지 생성
    prompt_result = await mcp.get_prompt(prompt_name, arguments=arguments)

    # 2. 생성된 메시지를 Claude API 형식으로 변환
    messages = [
        {
            "role": msg.role,
            "content": msg.content.text
        }
        for msg in prompt_result.messages
    ]

    # 3. Claude API 호출
    response = api_client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages
    )

    return response.content[0].text


# 구조 검토 프롬프트로 Claude 호출
result = await use_prompt_with_claude(
    "structural_review",
    {
        "member_type": "beam",
        "width": "300",
        "depth": "600",
        "fck": "27",
        "fy": "400"
    }
)
print(result)

## 핵심 정리

| MCP 기능 | 데코레이터 | 클라이언트 API | 제어 주체 |
|----------|-----------|---------------|----------|
| Tools | `@mcp.tool()` | `call_tool()` | 모델 |
| Resources | `@mcp.resource()` | `read_resource()` | 앱 |
| Prompts | `@mcp.prompt()` | `get_prompt()` | 사용자 |

**프롬프트 활용 패턴**:  
프롬프트 정의 → `get_prompt()`로 메시지 생성 → Claude API에 전달 → 응답 수신